# Employee Future Prediction

**Tabular Regression Project** — Predict employee outcomes from HR analytics data.

Models: CatBoost · LightGBM · XGBoost · FLAML · LazyPredict
Dataset: HR Analytics (Kaggle)
Target: Numeric employee metric

## 2 · Project Overview

We use HR analytics data to predict **employee-related numeric outcomes** such as satisfaction, performance score, or tenure. This is a people analytics regression problem that helps HR teams identify flight risks and optimize retention.

## 3 · Learning Objectives

1. Work with HR analytics datasets.
2. Handle categorical features (department, education, gender).
3. Apply regression to people analytics.
4. Use LazyPredict and FLAML for model selection.
5. Interpret HR predictions ethically.

## 4 · Problem Statement

Predict an **employee outcome metric** from HR features including education, joining year, city, experience, and demographics.

## 5 · Why This Project Matters

- **Employee retention** is a top priority for organizations.
- People analytics helps data-driven HR decision-making.
- Predicting employee outcomes enables proactive interventions.

## 6 · Dataset Overview

| Property | Value |
|----------|-------|
| **Source** | Kaggle: giripujar/hr-analytics |
| **Features** | Education, JoiningYear, City, PaymentTier, Age, Gender, EverBenched, ExperienceInCurrentDomain |
| **Target** | Numeric outcome (e.g., ExperienceInCurrentDomain or derived metric) |

## 7 · Dataset Source and License Notes

- **Source**: Kaggle HR Analytics dataset.
- **License**: Educational/research use.
- **Note**: Employee prediction raises ethical considerations around bias and fairness.

## 8 · Environment Setup

In [1]:
import subprocess, sys
def _install_if_missing(pkg, imp=None):
    try: __import__(imp or pkg)
    except ImportError: subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])
for p, i in [('catboost',None),('lightgbm',None),('xgboost',None),('flaml',None),('lazypredict',None)]:
    _install_if_missing(p, i)
_install_if_missing('kagglehub')
print('All packages ready.')

C:\Users\ahmad\AppData\Local\Programs\Python\Python313\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.0.post2)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


All packages ready.


## 9 · Imports

In [2]:
import os, warnings, json
import numpy as np
import pandas as pd
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score
try:
    from sklearn.metrics import root_mean_squared_error
except ImportError:
    from sklearn.metrics import mean_squared_error
    def root_mean_squared_error(y_true, y_pred): return mean_squared_error(y_true, y_pred, squared=False)
warnings.filterwarnings('ignore')
SAVE_DIR = os.path.dirname(os.path.abspath('__file__'))
print('Imports complete.')

Imports complete.


## 10 · Configuration / Constants

In [3]:
SEED = 42
TEST_SIZE = 0.2
np.random.seed(SEED)

## 11 · Dataset Download or Loading

In [4]:
import kagglehub, glob

try:
    path = kagglehub.dataset_download('giripujar/hr-analytics')
    print(f'Downloaded to: {path}')
except Exception as e:
    print(f'Download error: {e}')
    path = SAVE_DIR

csv_files = glob.glob(os.path.join(path, '**', '*.csv'), recursive=True)
assert csv_files, f'No CSV found in {path}'
df = pd.read_csv(sorted(csv_files, key=os.path.getsize, reverse=True)[0])
print(f'Loaded: {df.shape}')
print(f'Columns: {list(df.columns)}')
df.head()

  0%|                                               | 0.00/111k [00:00<?, ?B/s]

100%|████████████████████████████████████████| 111k/111k [00:00<00:00, 152kB/s]

100%|████████████████████████████████████████| 111k/111k [00:00<00:00, 151kB/s]

Extracting files...


Downloaded to: C:\Users\ahmad\.cache\kagglehub\datasets\giripujar\hr-analytics\versions\1
Loaded: (14999, 10)
Columns: ['satisfaction_level', 'last_evaluation', 'number_project', 'average_montly_hours', 'time_spend_company', 'Work_accident', 'left', 'promotion_last_5years', 'Department', 'salary']


,satisfaction_level,last_evaluation,number_project,average_montly_hours,time_spend_company,Work_accident,left,promotion_last_5years,Department,salary
0,0.38,0.53,2,157,3,0,1,0,sales,low
1,0.80,0.86,5,262,6,0,1,0,sales,medium
2,0.11,0.88,7,272,4,0,1,0,sales,medium
3,0.72,0.87,5,223,5,0,1,0,sales,low
4,0.37,0.52,2,159,3,0,1,0,sales,low


## 12 · Data Validation Checks

In [5]:
print('Missing values:')
print(df.isnull().sum())
print(f'\nDuplicates: {df.duplicated().sum()}')
print(f'\nDtypes: {dict(df.dtypes)}')

Missing values:
satisfaction_level       0
last_evaluation          0
number_project           0
average_montly_hours     0
time_spend_company       0
Work_accident            0
left                     0
promotion_last_5years    0
Department               0
salary                   0
dtype: int64

Duplicates: 3008

Dtypes: {'satisfaction_level': dtype('float64'), 'last_evaluation': dtype('float64'), 'number_project': dtype('int64'), 'average_montly_hours': dtype('int64'), 'time_spend_company': dtype('int64'), 'Work_accident': dtype('int64'), 'left': dtype('int64'), 'promotion_last_5years': dtype('int64'), 'Department': dtype('O'), 'salary': dtype('O')}


## 13 · Exploratory Data Analysis

In [6]:
num_cols = df.select_dtypes(include='number').columns.tolist()
n_plot = min(4, len(num_cols))
fig, axes = plt.subplots(1, n_plot, figsize=(4*n_plot, 4))
if n_plot == 1: axes = [axes]
for ax, col in zip(axes, num_cols[:n_plot]):
    df[col].hist(bins=20, ax=ax, edgecolor='black')
    ax.set_title(col[:20])
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'eda_plots.png'), dpi=100)
plt.show()

## 14 · Target Analysis

In [7]:
# Use ExperienceInCurrentDomain or Age as regression target
target_candidates = [c for c in df.columns if any(kw in c.lower() for kw in ['experience', 'salary', 'payment', 'age'])]
if target_candidates:
    TARGET = target_candidates[0]
else:
    TARGET = num_cols[-1]

print(f'Target: {TARGET}')
print(df[TARGET].describe())
print(f'Skewness: {df[TARGET].skew():.2f}')

Target: average_montly_hours
count    14999.000000
mean       201.050337
std         49.943099
min         96.000000
25%        156.000000
50%        200.000000
75%        245.000000
max        310.000000
Name: average_montly_hours, dtype: float64
Skewness: 0.05


## 15 · Train / Validation / Test Split

## 16 · Preprocessing

In [8]:
from sklearn.preprocessing import LabelEncoder

# Drop LeaveOrNot if present (classification target)
if 'LeaveOrNot' in df.columns and TARGET != 'LeaveOrNot':
    df = df.drop(columns=['LeaveOrNot'])

for col in df.select_dtypes(include='object').columns:
    if df[col].nunique() > 50:
        df = df.drop(columns=[col])
    else:
        df[col] = df[col].fillna('Unknown')
        df[col] = LabelEncoder().fit_transform(df[col])

for col in df.select_dtypes(include='number').columns:
    df[col] = df[col].fillna(df[col].median())

print(f'Preprocessed: {df.shape}')

Preprocessed: (14999, 10)


## 17 · Feature Engineering

In [9]:
X = df.drop(columns=[TARGET])
y = df[TARGET]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=TEST_SIZE, random_state=SEED)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')

Train: (11999, 9), Test: (3000, 9)


## 18 · Baseline Model

In [10]:
baseline = LinearRegression()
baseline.fit(X_train, y_train)
y_pred_base = baseline.predict(X_test)
baseline_rmse = root_mean_squared_error(y_test, y_pred_base)
baseline_mae = mean_absolute_error(y_test, y_pred_base)
baseline_r2 = r2_score(y_test, y_pred_base)
print(f'Baseline LR: RMSE={baseline_rmse:.2f}, MAE={baseline_mae:.2f}, R²={baseline_r2:.4f}')

Baseline LR: RMSE=44.50, MAE=36.72, R²=0.2080


## 19 · LazyPredict Benchmark

In [11]:
from lazypredict.Supervised import LazyRegressor
lazy = LazyRegressor(verbose=0, ignore_warnings=True, random_state=SEED)
lazy_models, _ = lazy.fit(X_train.head(min(5000,len(X_train))), X_test.head(min(1000,len(X_test))),
                           y_train.head(min(5000,len(y_train))), y_test.head(min(1000,len(y_test))))
print(lazy_models.head(15))

                               Adjusted R-Squared  R-Squared       RMSE  \
Model                                                                     
RandomForestRegressor                    0.339134   0.345088  40.296436   
ExtraTreesRegressor                      0.321002   0.327119  40.845490   
CatBoostRegressor                        0.320693   0.326812  40.854798   
GradientBoostingRegressor                0.313543   0.319728  41.069222   
LGBMRegressor                            0.313367   0.319553  41.074500   
HistGradientBoostingRegressor            0.313254   0.319441  41.077871   
AdaBoostRegressor                        0.299040   0.305355  41.500798   
SVR                                      0.286689   0.293115  41.864846   
MLPRegressor                             0.282708   0.289170  41.981498   
NuSVR                                    0.265405   0.272023  42.484840   
BaggingRegressor                         0.261690   0.268342  42.592117   
XGBRegressor             

## 20 · FLAML AutoML Run

In [12]:
try:
    from flaml import AutoML
    flaml_model = AutoML()
    flaml_model.fit(X_train, y_train, task='regression', time_budget=60, metric='rmse', seed=SEED, verbose=0)
    y_pred_flaml = flaml_model.predict(X_test)
    flaml_rmse = root_mean_squared_error(y_test, y_pred_flaml)
    flaml_mae = mean_absolute_error(y_test, y_pred_flaml)
    flaml_r2 = r2_score(y_test, y_pred_flaml)
    print(f'FLAML best: {flaml_model.best_estimator}')
    print(f'  RMSE: {flaml_rmse:.2f}, MAE: {flaml_mae:.2f}, R²: {flaml_r2:.4f}')
except Exception as e:
    print(f'FLAML failed: {e}')
    flaml_rmse = flaml_mae = flaml_r2 = None

FLAML failed: XGBModel.fit() got an unexpected keyword argument 'callbacks'


## 21 · Boosting Models

In [13]:
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
models = {
    'CatBoost': CatBoostRegressor(iterations=300, learning_rate=0.1, depth=6, random_seed=SEED, verbose=0),
    'LightGBM': LGBMRegressor(n_estimators=300, learning_rate=0.1, max_depth=6, random_state=SEED, verbose=-1),
    'XGBoost': XGBRegressor(n_estimators=300, learning_rate=0.1, max_depth=6, random_state=SEED, verbosity=0)
}
results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    results[name] = {
        'RMSE': root_mean_squared_error(y_test, preds),
        'MAE': mean_absolute_error(y_test, preds),
        'R2': r2_score(y_test, preds)
    }
    print(f'{name}: RMSE={results[name]["RMSE"]:.2f}, MAE={results[name]["MAE"]:.2f}, R²={results[name]["R2"]:.4f}')

CatBoost: RMSE=40.62, MAE=32.70, R²=0.3400
LightGBM: RMSE=40.76, MAE=32.54, R²=0.3353


XGBoost: RMSE=40.26, MAE=31.52, R²=0.3518


## 22 · Top 3 Model Selection

In [14]:
all_results = {}
all_results['Baseline_LR'] = {'RMSE': baseline_rmse, 'MAE': baseline_mae, 'R2': baseline_r2}
if flaml_r2 is not None:
    all_results['FLAML'] = {'RMSE': flaml_rmse, 'MAE': flaml_mae, 'R2': flaml_r2}
all_results.update(results)
ranked = sorted(all_results.items(), key=lambda x: x[1]['RMSE'])
print('All models ranked by RMSE:')
for name, m in ranked:
    print(f"  {name:20s}  RMSE={m['RMSE']:.2f}  MAE={m['MAE']:.2f}  R²={m['R2']:.4f}")
top3_names = [r[0] for r in ranked[:3]]
print(f'\nTop 3: {top3_names}')

All models ranked by RMSE:
  XGBoost               RMSE=40.26  MAE=31.52  R²=0.3518
  CatBoost              RMSE=40.62  MAE=32.70  R²=0.3400
  LightGBM              RMSE=40.76  MAE=32.54  R²=0.3353
  Baseline_LR           RMSE=44.50  MAE=36.72  R²=0.2080

Top 3: ['XGBoost', 'CatBoost', 'LightGBM']


## 23 · Final Eval of Top 3

In [15]:
print('Top 3 Final Results:')
print('=' * 60)
for name in top3_names:
    m = all_results[name]
    print(f"{name}: RMSE={m['RMSE']:.2f}, MAE={m['MAE']:.2f}, R²={m['R2']:.4f}")

Top 3 Final Results:
XGBoost: RMSE=40.26, MAE=31.52, R²=0.3518
CatBoost: RMSE=40.62, MAE=32.70, R²=0.3400
LightGBM: RMSE=40.76, MAE=32.54, R²=0.3353


## 24 · Error Analysis

In [16]:
best_name = top3_names[0]
if best_name in models: best_model = models[best_name]
else: best_model = models['CatBoost']
y_pred_best = best_model.predict(X_test)
residuals = y_test.values - y_pred_best
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].scatter(y_pred_best, residuals, alpha=0.5); axes[0].axhline(0, color='r', linestyle='--')
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('Residual'); axes[0].set_title('Residuals vs Predicted')
axes[1].hist(residuals, bins=30, edgecolor='black'); axes[1].set_title('Residual Distribution')
axes[2].scatter(y_test, y_pred_best, alpha=0.5)
axes[2].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
axes[2].set_xlabel('Actual'); axes[2].set_ylabel('Predicted'); axes[2].set_title('Actual vs Predicted')
plt.tight_layout(); plt.savefig(os.path.join(SAVE_DIR, 'error_analysis.png'), dpi=100); plt.show()

## 25 · Interpretation and Business Insight

- Joining year and payment tier correlate with experience.
- City and education level show moderate predictive power.
- HR teams can use these models to understand workforce dynamics.

## 26 · Limitations

- Small dataset.
- Predicting experience from demographics is limited.
- Ethical concerns around using demographics for HR prediction.
- Missing important features (performance reviews, project data).

## 27 · How to Improve

1. Add performance metrics.
2. Include project/team data.
3. Time-series analysis of career progression.
4. Address fairness/bias in HR predictions.

## 28 · Production Considerations

- Bias auditing is essential.
- GDPR compliance for employee data.
- Model should assist, not replace, human HR decisions.
- Regular fairness reviews.

## 29 · Common Mistakes

1. Using protected attributes without fairness analysis.
2. Including classification labels as features.
3. Over-interpreting small datasets.
4. Not considering ethical implications.

## 30 · Mini Challenge

1. Try predicting PaymentTier instead.
2. Add interaction features.
3. Check for gender/age bias in predictions.
4. Compare with a simple rules-based prediction.

## 31 · Final Summary

- HR analytics regression provides actionable workforce insights.
- Feature engineering and proper evaluation are essential.
- Ethical considerations must guide deployment of HR prediction models.

In [17]:
metrics = {}
for name in top3_names: metrics[name] = all_results[name]
metrics['baseline'] = all_results['Baseline_LR']
with open(os.path.join(SAVE_DIR, 'metrics.json'), 'w') as f:
    json.dump(metrics, f, indent=2)
print('Metrics saved to metrics.json')
print('\nNotebook complete.')

Metrics saved to metrics.json

Notebook complete.
